In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [6]:
df=pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])

In [7]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [8]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [9]:
unique_emotions=df['emotion'].unique()
emotion_numbers={}
i=0
for emo in unique_emotions:
   emotion_numbers[emo]=i
   i+=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [35]:
unique_emotions

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [11]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [12]:
df['text']=df['text'].apply(lambda x : x.lower())
df['text']

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15995    i just had a very brief time in the beanbag an...
15996    i am now turning and i feel pathetic that i am...
15997                       i feel strong and good overall
15998    i feel like this was such a rude comment and i...
15999    i know a lot but i feel so stupid because i ca...
Name: text, Length: 16000, dtype: object

In [13]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [14]:
df['text']=df['text'].apply(remove_punc)

In [15]:
def remove_numbers(txt):
  new=''
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new

df['text'] = df['text'].apply(remove_numbers)

In [16]:
def remove_emojis(txt):
  new=''
  for i in txt:
    if i.isascii():
      new+=i
  return new

df['text']=df['text'].apply(remove_emojis)



In [17]:
import nltk

In [18]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jaswi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jaswi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [19]:
stop_words=set(stopwords.words('english'))
def remove_stopwords(txt):
  word_tokens = txt.split()
  cleaned=[]
  for i in word_tokens:
    if i not in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)

df['text']=df['text'].apply(remove_stopwords)


In [20]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [21]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df['text'],df['emotion'],test_size=0.2,random_state=42)

In [22]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((12800,), (3200,), (12800,), (3200,))

In [23]:
df.shape

(16000, 2)

In [24]:
#vectorization of text by bag of words
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
bow_vectorizer = CountVectorizer()
x_train_bow = bow_vectorizer.fit_transform(x_train)
x_test_bow = bow_vectorizer.transform(x_test)


In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [26]:
nb_model=MultinomialNB()
nb_model.fit(x_train_bow,y_train)
y_pred=nb_model.predict(x_test_bow)

In [27]:
print(accuracy_score(y_test,y_pred))

0.768125


In [28]:
#tfidf
vectorizer = TfidfVectorizer()
x_train_vectorized = vectorizer.fit_transform(x_train)
x_test_vectorized = vectorizer.transform(x_test)

In [29]:
nb2_model=MultinomialNB()
nb2_model.fit(x_train_vectorized,y_train)
y_pred2=nb2_model.predict(x_test_vectorized)

In [30]:
print(accuracy_score(y_test,y_pred2))

0.6609375


In [31]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression(max_iter=1000)
lr_model.fit(x_train_vectorized,y_train)
y_pred3=lr_model.predict(x_test_vectorized)

In [32]:
print(accuracy_score(y_test,y_pred3))

0.8628125


In [33]:
import joblib
joblib.dump(lr_model,'lr_model.pkl')
joblib.dump(vectorizer,'vectorizer.pkl')

['vectorizer.pkl']

In [34]:
from google.colab import files
files.download('lr_model.pkl')
files.download('vectorizer.pkl')
files.download('sentimentanlysis.ipynb')

ModuleNotFoundError: No module named 'google.colab'